# 03 · Finding signals: FCME

**Question:** in a spectrum, which bins are signals and which are noise?

To answer it you need the **noise floor**. Averaging all bins does not give it: the signals pull the average up, and a band full of stations can lift it by many dB. Then weak signals fall under a threshold set on that inflated "floor" (self-masking).

**FCME (forward consecutive mean excision)** avoids this:
1. start with only the quietest bins (say the lowest 25 %) as "noise",
2. take their mean, set a threshold `T × mean`,
3. every bin under the threshold joins the noise set,
4. repeat until the set stops changing. Bins above the final threshold are signals.

**T comes from statistics.** One bin's power, averaged over K FFTs, follows a Gamma(K) distribution around the true floor. For a chosen false-alarm probability per bin (`pfa`), `T = gamma.ppf(1 − pfa, K) / K`. More averages → tighter noise → smaller T.

**FCME assumes a flat floor.** Real floors are not: the HackRF's filter shape and DC spike (receiver profile, notebook 02) and slow tilts from the antenna. So Tayf first divides out the receiver shape, removes a slow trend, and excludes the DC bins. And because ~250 averages make the statistical T only ~1 dB, a practical minimum SNR (6 dB) is used for detection, with a lower edge threshold (3 dB) to find where each signal starts and ends.

Change the parameters and *Run All*.

In [ ]:
# ---- Parameters: change these ----
K = 200                    # FFT averages per spectrum (Tayf's live frames: ~100-250)
N_BINS = 2048
FLOOR_DBFS = -80.0
TILT_DB = 6.0              # floor rises this much from left to right (0 = flat)
SIGNALS = [                # (offset kHz, width kHz, SNR dB); width 0 = a tone
    (-600, 0, 10.0), (-300, 0, 4.0), (100, 200, 8.0), (650, 20, 15.0),
]
PFA = 1e-3
MIN_SNR_DB = 6.0
EDGE_SNR_DB = 3.0
SEED = 0

In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import gamma
from tayf.core.models import SpectrumFrame, utc_now
from tayf.dsp.detect import DetectorConfig, detect, fcme, threshold_factor

DF = 1e3
rng = np.random.default_rng(SEED)
f = (np.arange(N_BINS) - N_BINS // 2) * DF
floor_true = FLOOR_DBFS + TILT_DB * (np.arange(N_BINS) / N_BINS - 0.5)
x = 10 ** (floor_true / 10) * rng.gamma(K, 1 / K, N_BINS)
for off, width, snr in SIGNALS:
    m = np.abs(f - off * 1e3) <= max(width * 1e3 / 2, DF / 2)
    x[m] += 10 ** ((floor_true[m] + snr) / 10)
p_db = 10 * np.log10(x)
frame = SpectrumFrame(f, p_db.astype(np.float32), DF, 0.0, utc_now(), n_averages=K)
print(f"{N_BINS} bins, K = {K} averages; true floor {FLOOR_DBFS} dBFS/bin with a {TILT_DB} dB tilt")

## 1 · Why not just average?
The plain mean of all bins, the median, and FCME's estimate, against the true floor in the middle of the band.

In [ ]:
fac = threshold_factor(PFA, K)
mu, clean = fcme(x / 10 ** ((floor_true - FLOOR_DBFS) / 10), fac)      # (on the de-tilted spectrum, for this comparison)
print(f"true floor (middle)       {FLOOR_DBFS:7.2f} dBFS")
print(f"mean of all bins          {10*np.log10(x.mean()):7.2f} dBFS   <- pulled up by the signals")
print(f"median of all bins        {np.median(p_db):7.2f} dBFS")
print(f"FCME noise mean           {10*np.log10(mu):7.2f} dBFS   ({clean.mean()*100:.0f} % of bins judged noise)")

## 2 · The threshold factor T
How far above the noise mean averaged noise still reaches, with probability `pfa`. With K = 1 (no averaging) noise needs a big margin; with hundreds of averages, about 1 dB.

In [ ]:
ks = [1, 2, 5, 10, 20, 50, 100, 200, 500]
fig = go.Figure()
for pfa in (1e-2, 1e-3, 1e-4, 1e-6):
    fig.add_scatter(x=ks, y=[10 * np.log10(threshold_factor(pfa, k)) for k in ks], name=f"pfa {pfa:g}", mode="lines+markers")
fig.add_hline(y=MIN_SNR_DB, line_dash="dash", annotation_text=f"practical minimum ({MIN_SNR_DB} dB)")
fig.update_layout(height=350, xaxis_type="log", xaxis_title="averages K", yaxis_title="threshold over the noise mean (dB)",
                  title="Statistical threshold vs averaging", margin=dict(t=50))
fig.show()
print(f"K = {K}, pfa = {PFA:g}: T = {10*np.log10(fac):.2f} dB  ->  detection threshold used = max(T, {MIN_SNR_DB}) dB")

## 3 · FCME, iteration by iteration
The noise set (dots) grows from the quietest bins until it stops changing. Shown on the de-tilted spectrum.

In [ ]:
xd = x / 10 ** ((floor_true - FLOOR_DBFS) / 10)
xs = np.sort(xd)
clean_i = xd <= xs[int(len(xs) * 0.25) - 1]
fig = go.Figure()
fig.add_scatter(x=f / 1e3, y=10 * np.log10(xd), name="spectrum", line=dict(width=1, color="#888"))
for it in range(6):
    m = xd[clean_i].mean()
    fig.add_hline(y=10 * np.log10(fac * m), line=dict(width=1, dash="dot"),
                  annotation_text=f"iter {it}: {clean_i.mean()*100:.0f} % noise", annotation_position="top left")
    new = xd <= fac * m
    if np.array_equal(new, clean_i):
        break
    clean_i = new
fig.add_scatter(x=f[clean_i] / 1e3, y=10 * np.log10(xd[clean_i]), mode="markers", marker=dict(size=2), name="noise set (final)")
fig.update_layout(height=380, xaxis_title="offset (kHz)", yaxis_title="dBFS / bin", margin=dict(t=30))
fig.show()

## 4 · The full detector
Trend removal, FCME, local floor, then detections with edges (boxes). Compare with the truth (dashed).

In [ ]:
cfg = DetectorConfig(pfa=PFA, min_snr_db=MIN_SNR_DB, edge_snr_db=EDGE_SNR_DB)
dets, info = detect(frame, cfg)
fig = go.Figure()
fig.add_scatter(x=f / 1e3, y=p_db, name="spectrum", line=dict(width=1))
fig.add_scatter(x=f / 1e3, y=info["floor_db"], name="estimated floor", line=dict(width=2))
fig.add_scatter(x=f / 1e3, y=floor_true, name="true floor", line=dict(width=1, dash="dash"))
fig.add_scatter(x=f / 1e3, y=info["floor_db"] + info["threshold_db"], name="detection threshold", line=dict(width=1, dash="dot"))
for d in dets:
    fig.add_vrect(x0=d.f_lo_hz / 1e3, x1=d.f_hi_hz / 1e3, fillcolor="orange", opacity=0.25, line_width=0)
fig.update_layout(height=420, xaxis_title="offset (kHz)", yaxis_title="dBFS / bin", margin=dict(t=30))
fig.show()
print(f"{'found at (kHz)':>15} {'width (kHz)':>12} {'SNR (dB)':>9}")
for d in dets:
    print(f"{d.f_center_hz/1e3:>15.1f} {d.bandwidth_hz/1e3:>12.1f} {d.snr_db:>9.1f}")
print("truth:", [(o, w, s) for o, w, s in SIGNALS], "  (offset kHz, width kHz, SNR dB)")

**Read it:** signals under `MIN_SNR_DB` are not found (try the 4 dB tone). A wide signal's measured width is where it stands at least `EDGE_SNR_DB` over the floor; for weak wide signals that is narrower than its true width. The Signal Inspector (next step) measures occupied bandwidth properly.

## 5 · Detection vs SNR (the "cliff")
How often a single tone is found, against its SNR, for a few thresholds, plus false alarms in pure noise. The threshold, not the algorithm, decides where the cliff is (the old SpectraIQ finding).

Two things to notice:
- The tone's SNR here is its power over the noise power in its bin. What the spectrum shows is tone + noise, i.e. 10·log10(1 + SNR) above the floor: a 5 dB tone stands about 6 dB up, a 0 dB tone 3 dB.
- With K averaged FFTs the noise in a bin hardly moves (±0.3 dB at K = 200), so the cliff is only ~1 dB wide and even a 3 dB threshold gives no false alarms *on ideal noise*. Real spectra have floor ripple, spurs and interference left over, which is why Tayf uses 6 dB. Try `K = 2` to see a soft cliff and false alarms.

In [ ]:
snrs = np.arange(0, 14, 1.0)
trials = 60
fig = go.Figure()
for thr in (3.0, 6.0, 9.0):
    c = DetectorConfig(pfa=PFA, min_snr_db=thr, edge_snr_db=min(EDGE_SNR_DB, thr))
    pd_ = []
    for s in snrs:
        hit = 0
        for t in range(trials):
            r = np.random.default_rng(1000 + t)
            xx = 10 ** (FLOOR_DBFS / 10) * r.gamma(K, 1 / K, N_BINS)
            xx[N_BINS // 2 + 300] += 10 ** ((FLOOR_DBFS + s) / 10)
            fr = SpectrumFrame(f, (10 * np.log10(xx)).astype(np.float32), DF, 0.0, utc_now(), n_averages=K)
            hit += any(abs(d.f_center_hz - 300e3) < 3 * DF for d in detect(fr, c)[0])
        pd_.append(hit / trials)
    fa = sum(len(detect(SpectrumFrame(f, (10*np.log10(10**(FLOOR_DBFS/10)*np.random.default_rng(t).gamma(K,1/K,N_BINS))).astype(np.float32),
                                      DF, 0.0, utc_now(), n_averages=K), c)[0]) for t in range(trials))
    fig.add_scatter(x=snrs, y=pd_, mode="lines+markers", name=f"min SNR {thr:g} dB  ({fa} false alarms in {trials} noise spectra)")
fig.update_layout(height=380, xaxis_title="tone SNR in its bin (dB)", yaxis_title="probability of detection",
                  margin=dict(t=30))
fig.show()

## 6 · On a real recording
Pick a recording from `data/recordings`. When it was made with this HackRF at a profiled sample rate, the receiver's floor shape is divided out first, the DC bins are skipped, and the HackRF's own carriers are flagged as spurs.

In [ ]:
RECORDING = "fm.bin.sigmf-meta"         # any file in data/recordings
TIME_S = 2.0
NFFT = 4096
USE_PROFILE = True

In [ ]:
from pathlib import Path
from tayf.sources.recording import RecordingSource
from tayf.dsp.dc import remove_dc
from tayf.dsp.spectrum import spectrum_frame
from tayf.dsp.detect import shape_for
from tayf.receiver.profile import load_profile
import tayf

root = Path(tayf.__file__).resolve().parents[1]
prof = load_profile(root / "data" / "receiver") if USE_PROFILE else None
src = RecordingSource(root / "data" / "recordings" / RECORDING)
blk = src.read_at(TIME_S, 262144)
blk.samples = remove_dc(blk.samples)
fr = spectrum_frame(blk, NFFT)
shape = shape_for(prof, fr)
spurs = [c["freq_hz"] for c in (prof or {}).get("internal_carriers", [])]
dets, info = detect(fr, DetectorConfig(pfa=PFA, min_snr_db=MIN_SNR_DB, edge_snr_db=EDGE_SNR_DB),
                    shape_db=shape, spurs_hz=spurs, lo_hz=blk.center_freq_hz)
fig = go.Figure()
fig.add_scatter(x=fr.freqs_hz / 1e6, y=fr.power_dbfs, name="spectrum", line=dict(width=1))
fig.add_scatter(x=fr.freqs_hz / 1e6, y=info["floor_db"] + (shape if shape is not None else 0), name="floor", line=dict(width=2))
for d in dets:
    fig.add_vrect(x0=d.f_lo_hz / 1e6, x1=d.f_hi_hz / 1e6, fillcolor="red" if d.spur else "orange", opacity=0.25, line_width=0)
fig.update_layout(height=420, xaxis_title="MHz", yaxis_title="dBFS / bin", margin=dict(t=30),
                  title=f"{RECORDING} at {TIME_S} s · {len(dets)} detections · receiver shape {'used' if shape is not None else 'not available'}")
fig.show()
for d in sorted(dets, key=lambda d: d.f_center_hz):
    print(f"{d.f_center_hz/1e6:10.4f} MHz  width {d.bandwidth_hz/1e3:6.1f} kHz  SNR {d.snr_db:5.1f} dB" + ("  (HackRF spur)" if d.spur else ""))

## Try it yourself
- Set `TILT_DB = 15` and compare the estimated floor with the true one.
- Set `K = 1` (a single FFT): how does the statistical threshold compare with `MIN_SNR_DB`, and how many false alarms appear in section 5?
- Lower `MIN_SNR_DB` to 3 dB: the cliff moves left, but count the false alarms.
- Try `RECORDING = "synthetic_scene.sigmf-meta"` (no profile shape: it was not recorded by the HackRF) and compare with its true signals.